<a href="https://colab.research.google.com/github/liminalvoid/nlp/blob/main/sem_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 7. Fine-tuning BERT

## Установка, импорт библиотек и базовые настройки

In [ ]:
%pip -q install -U transformers datasets evaluate accelerate seqeval scikit-learn

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    DataCollatorWithPadding,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
    default_data_collator,
)


SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)
if torch.cuda.is_available():
    print("GPU model:", torch.cuda.get_device_name(0))

## Sequence classification

В качестве исходного датасета используется набор данных для классификации новостных текстов на русском ([data-silence/rus_news_classifiera](https://huggingface.co/datasets/data-silence/rus_news_classifier)).

In [ ]:
dataset_name = "data-silence/rus_news_classifier"
dataset = load_dataset(dataset_name)

print(dataset)

for split in ["train", "test"]:
    print(f"\n--- {split} examples ---")

    for i in range(3):
        example = dataset[split][i]

        print(
            f"[{i}] sentence = {example["news"]}"
            f"    label    = {example["labels"]}"
        )

Ниже представлена токенизация и препроцессинг исходного корпуса.

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
preprocess = lambda batch: tokenizer(
    batch["news"],
    truncation=True,
    max_length=128,
)

tokenized_dataset = dataset.map(preprocess, batched=True)

tokenized_dataset

Загрузка BERT с classification head и вывод модели.

In [ ]:
label2id = {
    "climate": 0,
    "conflicts": 1,
    "culture": 2,
    "economy": 3,
    "gloss": 4,
    "health": 5,
    "politics": 6,
    "science": 7,
    "society": 8,
    "sports": 9,
    "travel": 10,
}
id2label = {v: k for k, v in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

print(model)
print("\nClassifier head:")
print(model.classifier)

Метрики для оценки результата.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def calculate_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(
        predictions=preds,
        references=labels,
    )["accuracy"]
    f1 = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="weighted",
    )["f1"]

    return {"accuracy": acc, "f1": f1}

Fine-tuning на исходном датасете.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

selected_range = range(800)
train_dataset = tokenized_dataset["train"].select(selected_range)
test_dataset = tokenized_dataset["test"].select(selected_range)

steps_per_epoch = len(train_dataset) // 16
eval_steps = 5

print("Приблизительное количество шагов за эпоху:", steps_per_epoch)
print("Валидация каждые", eval_steps, "шагов")

training_args = TrainingArguments(
    output_dir="output",
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    logging_strategy="steps",
    logging_steps=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=calculate_metrics,
)

trainer.train()

Оценка на тестовой выборке.

In [ ]:
metrics = trainer.evaluate()
metrics

Отфильтрованные логи и графики.

In [ ]:
log_df = pd.DataFrame(trainer.state.log_history)

train_logs = log_df.dropna(subset=["loss"]).copy() if "loss" in log_df.columns else pd.DataFrame()
test_logs = log_df.dropna(subset=["eval_loss"]).copy() if "eval_loss" in log_df.columns else pd.DataFrame()
summary_logs = log_df.dropna(subset=["train_loss"]).copy() if "train_loss" in log_df.columns else pd.DataFrame()

print("=== Train Logs ===")
if len(train_logs) > 0:
    cols = [
        c for c in ["step", "epoch", "loss", "grad_norm", "learning_rate"]
        if c in train_logs.columns
    ]
    display(train_logs[cols].tail(10))
else:
    print("Нет train-логов")

print("\n=== Eval Logs ===")
if len(test_logs) > 0:
    cols = [
        c for c in ["step", "epoch", "eval_loss", "eval_accuracy", "eval_f1", "eval_runtime"]
        if c in test_logs.columns
    ]
    display(test_logs[cols].tail(10))
else:
    print("Нет train-логов")

print("\n=== Summary Logs ===")
if len(summary_logs) > 0:
    cols = [
        c for c in ["train_runtime", "train_samples_per_second", "train_steps_per_second", "train_loss", "epoch"]
        if c in summary_logs.columns
    ]
else:
    print("Нет summary-логов")

plt.figure(figsize=(8, 5))

if len(train_logs) > 0:
    plt.plot(train_logs["step"], train_logs["loss"], marker="o", label="Train loss")

if len(test_logs) > 0:
    plt.plot(test_logs["step"], test_logs["eval_loss"], marker="o", label="Test loss")

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Fine-tuning: filtered loss curves")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

metric_cols = [
    c for c in ["eval_accuracy", "eval_f1"]
    if c in test_logs.columns
]

if len(test_logs) > 0 and len(metric_cols) > 0:
    plt.figure(figsize=(8, 5))

    for col in metric_cols:
        plt.plot(
            test_logs["step"],
            test_logs[col],
            marker="o",
            label=col.replace("eval_", ""),
        )

    plt.xlabel("Step")
    plt.ylabel("Score")
    plt.title("Fine-tuning: test metrics")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("Нет test-метрик для построения графика")

Предсказания модели.

In [ ]:
sample_texts = [
    "В Перми скончался 82-летний актер театра и кино Всевлод Авсентьев",
    "Чемпионат мира по футболу закончился победой Румынии",
    "В 2021 году Microsoft выпустит специальную версию Windows для бюджетных компьютеров.",
]
inputs = tokenizer(
    sample_texts,
    truncation=True,
    padding=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=-1).cpu().numpy()

for text, pred in zip(sample_texts, preds):
    print(f"TEXT: {text}")
    print(f"PRED: {id2label[int(pred)]}")
    print()

## Token classification / NER

Для этого задания используется датасет [unimelb-nlp/wikiann](https://huggingface.co/datasets/unimelb-nlp/wikiann).

In [ ]:
dataset = load_dataset("unimelb-nlp/wikiann", "ru")
print(dataset)

labels = dataset["train"].features["ner_tags"].feature.names
print("NER labels:", labels)

for i in range(2):
    ex = dataset["train"][i]
    print(f"\n--- example {i}")
    print("tokens:", ex["tokens"])
    print("tags:  ", [labels[t] for t in ex["ner_tags"]])

Токенизатор и выравнивание NER-меток по subwords.

In [ ]:
ner_checkpoint = "bert-base-cased"
ner_tokenizer = AutoTokenizer.from_pretrained(ner_checkpoint)


def tokenize_and_align_labels(examples):
    tokenized = ner_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels

    return tokenized


ner_tokenized = dataset.map(tokenize_and_align_labels, batched=True)
print(ner_tokenized)

Загрузка BERT с token classification head и печать модели.

In [ ]:
num_labels = len(labels)

model = AutoModelForTokenClassification.from_pretrained(
    ner_checkpoint,
    num_labels=num_labels,
    id2label={i: name for i, name in enumerate(labels)},
    label2id={name: i for i, name in enumerate(labels)},
)

print(model)
print("\nClassifier head:")
print(model.classifier)

Метрики для NER.

In [ ]:
label_names = [model.config.id2label[i] for i in range(len(model.config.id2label))]
sequeval = evaluate.load("seqeval")


def compute_metrics_ner(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        cur_preds = []
        cur_labels = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                cur_preds.append(label_names[p])   # map id -> string
                cur_labels.append(label_names[l])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    metrics = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": metrics["overall_precision"],
        "recall":    metrics["overall_recall"],
        "f1":        metrics["overall_f1"],
        "accuracy":  metrics["overall_accuracy"],
    }

Fine-tuning на исходном датасете.

In [ ]:
ner_data_collator = DataCollatorForTokenClassification(tokenizer=ner_tokenizer)
ner_training_args = TrainingArguments(
    output_dir="./bert-wiki",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
)
ner_trainer = Trainer(
    model=model,
    args=ner_training_args,
    train_dataset=ner_tokenized["train"],
    eval_dataset=ner_tokenized["validation"],
    processing_class=ner_tokenizer,
    data_collator=ner_data_collator,
    compute_metrics=compute_metrics_ner,
)

ner_trainer.train()

Оценка на validation.

In [ ]:
ner_metrics = ner_trainer.evaluate()
ner_metrics

Несколько предсказаний NER.

In [ ]:
example = dataset["validation"][0]
tokens = example["tokens"]

inputs = ner_tokenizer(
    tokens,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True,
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    pred_ids = outputs.logits.argmax(dim=-1)[0].cpu().tolist()

word_ids = inputs.word_ids(batch_index=0)

predicted_labels = []
seen = set()

for token_idx, word_id in enumerate(word_ids):
    if word_id is None or word_id in seen:
        continue

    seen.add(word_id)
    predicted_labels.append((tokens[word_id], label_names[pred_ids[token_idx]]))

print("Token -> predicted label")
for token, label in predicted_labels:
    print(f"{token:15s} {label}")

Вопросы для самопроверки:

1. Почему для классификации текста удобно использовать `[CLS]`-представление?  
Токен `[CLS]` специально добавляется в начало каждой последовательности при обучении BERT. За счёт механизма self-attention его итоговое скрытое состояние «видит» все остальные токены и агрегирует информацию о всём предложении — то есть получается вектор фиксированной длины, описывающий весь вход целиком. Это удобно для классификации: вместо того чтобы как-то усреднять или пулить представления всех токенов, мы берём один вектор `[CLS]`, подаём его в линейный слой-классификатор и получаем логиты по классам. К тому же во время предобучения (в задаче NSP у оригинального BERT) `[CLS]` уже учился представлять предложение целиком, так что это естественная «точка входа» для downstream-классификации.
2. Чем отличаются `AutoModelForSequenceClassification` и `AutoModelForTokenClassification`?  
Оба класса — это обёртки над одним и тем же базовым трансформером, но с разными «головами» сверху и разной формой выхода:

   - `AutoModelForSequenceClassification` решает задачу «одна метка на всю последовательность» (sentiment analysis, NLI, классификация тем и т.д.). Голова берёт представление `[CLS]` (точнее — pooled output) и проецирует его в вектор размерности `num_labels`. Логиты имеют форму `(batch_size, num_labels)`. Лейблы при обучении — один int на пример;
   - `AutoModelForTokenClassification` решает задачу «метка на каждый токен» (NER, POS-теггинг, chunking). Голова применяется к скрытым состояниям всех токенов, а не только `[CLS]`. Логиты имеют форму `(batch_size, seq_len, num_labels)`. Лейблы — последовательность той же длины, что и вход, по одной метке на токен;  
   
    То есть разница не в backbone, а в том, к чему прикладывается классификационный слой и какой размерности получается выход.
3. Зачем в NER мы используем `-100`?  
`-100` — это специальное значение, которое по умолчанию игнорирует `CrossEntropyLoss` в PyTorch (параметр `ignore_index=-100`). В NER оно нужно из-за несоответствия между «словами» и «субтокенами»:

    - Разметка NER даётся на уровне слов, а BERT-токенизатор режет слова на sub-word pieces (например, `Washington → Wash`, `##ington`);
    - Для первого субтокена слова мы ставим настоящую метку (`B-LOC`), а для всех последующих субтокенов того же слова — `-100`, чтобы они не участвовали в подсчёте лосса и метрик;
    - Аналогично `-100` ставится для служебных токенов: `[CLS]`, `[SEP]`, `[PAD]`.

    Благодаря этому модель обучается предсказывать метку только один раз на слово, и паддинги/служебные токены не искажают градиенты и метрики. Именно поэтому в `compute_metrics` нужно вручную отфильтровать все позиции с `l == -100` перед передачей в `seqeval` — иначе метрика увидит «мусорные» позиции.

4. Почему для NER часто лучше `bert-base-cased`?  
Потому что для распознавания именованных сущностей регистр букв — это сильнейший признак. В английском с заглавной буквы пишутся именно имена людей, названия организаций, географические объекты и т.п. Сравните: `apple` (фрукт) vs `Apple` (компания), `bush` (куст) vs `Bush` (фамилия):  
    - `bert-base-uncased` приводит всё к нижнему регистру ещё на этапе токенизации — эта информация теряется **безвозвратно**, и модель физически не может её использовать.
    - `bert-base-cased` сохраняет регистр, и модель учится использовать заглавные буквы как очень полезный сигнал для границ и типов сущностей.

    На классических NER-бенчмарках (CoNLL-2003) cased-версия стабильно даёт выше F1, чем uncased, примерно на 1–2 пункта. Для задач вроде sentiment analysis разница обычно незначительна или даже в пользу uncased, а вот для NER cased — почти всегда правильный выбор.

5. Что меняется, если взять другой датасет, но ту же задачу?  
Если задача остаётся та же (например, NER), большая часть пайплайна переиспользуется без изменений: архитектура модели, функция выравнивания меток с субтокенами, обработка `-100`, `compute_metrics` на основе `seqeval`, `DataCollatorForTokenClassification`, цикл обучения. Меняется в основном следующее:  
    - Набор меток и их количество. У CoNLL-2003 — 9 меток (`O`, `B-PER`, `I-PER`, `B-ORG`, …), у OntoNotes — 37, у биомедицинских датасетов вроде BC5CDR — свои (`B-Chemical`, `B-Disease`). Значит, нужно заново построить `label2id` / `id2label` и пересоздать модель с правильным `num_labels` — старую голову классификатора переиспользовать нельзя, backbone же переиспользуется;
    - Схема разметки. Где-то IOB2 (`B-`/`I-`/`O`), где-то BIOES/IOBES с дополнительными `E-` и `S-`. `seqeval` это поддерживает, но надо следить за согласованностью;
    - Формат самих данных. Поля могут называться по-разному (`tokens` / `words`, `ner_tags` / `labels` / `tags`), разметка может быть на уровне слов или уже на уровне символов (тогда нужна конвертация в BIO);
    - Домен и язык. Для биомедицины лучше взять `BioBERT` или `PubMedBERT`, для юридических текстов — `LegalBERT`, для русского — `rubert-base-cased` или мультиязычную модель. Общий `bert-base-cased` на узкоспециальной лексике будет работать заметно хуже;
    - Гиперпараметры. Число эпох, learning rate, размер батча и max_length стоит подбирать под размер и длину примеров нового датасета;
    - Метрики остаются теми же (precision/recall/F1 на уровне сущностей через `seqeval`), но абсолютные значения несравнимы между датасетами — CoNLL «лёгкий», биомед и соцсети — заметно сложнее.

## Sentence Pair Classification

Для задачи SPC использован датасет glue.

In [ ]:
dataset = load_dataset("glue", "qnli")
print(dataset)

for split in ["train", "validation"]:
    print(f"\n--- {split} examples ---")

    for i in range(2):
        ex = dataset[split][i]
        print(f"[{i}] question = {ex["question"]}")
        print(f"    sentence = {ex["sentence"]}")
        print(f"    label    = {ex["label"]}")
        print()

Токенизация пар предложений.

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def preprocess_pair(batch):
    return tokenizer(
        batch["question"],
        batch["sentence"],
        truncation=True,
        max_length=256,
    )


train_small = dataset["train"].select(range(3000))
val_small = dataset["validation"].select(range(500))

train_tok = train_small.map(preprocess_pair, batched=True)
val_tok = val_small.map(preprocess_pair, batched=True)
print(train_tok)
print(val_tok)

Загрузка BERT с головой для SPC.

In [ ]:
id2label = {
    idx: label
    for idx, label in enumerate(dataset["train"].features["label"].names)
}
label2id = {label: idx for idx, label in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

print(model)
print("\nClassifier head:")
print(model.classifier)

Метрики и fine-tuning.

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")


def compute_metrics_pair(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(
        predictions=preds,
        references=labels,
    )["accuracy"]
    f1_metric = f1.compute(
        predictions=preds,
        references=labels,
        average="binary",
    )["f1"]

    return {"accuracy": acc, "f1": f1_metric}


collator = DataCollatorWithPadding(tokenizer=tokenizer)
args = TrainingArguments(
    output_dir="./bert-mrpc",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics_pair,
)

trainer.train()
metrics = trainer.evaluate()
metrics

Предсказания для SPC.

In [ ]:
i, j = 1337, 228
examples = [
    (dataset["test"][i]["question"], dataset["test"][i]["sentence"]),
    (dataset["test"][j]["question"], dataset["test"][j]["sentence"]),
]

enc = tokenizer(
    [x[0] for x in examples],
    [x[1] for x in examples],
    truncation=True,
    padding=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    logits = model(**enc).logits
    preds = logits.argmax(dim=-1).cpu().tolist()

for (question, sentence), pred in zip(examples, preds):
    print("QUESTION:", question)
    print("SENTENCE:", sentence)
    print("PRED    :", id2label[pred])

## Extractive Question Answering

Для данного задания используется датасет squad_v2.

In [ ]:
dataset = load_dataset("squad_v2")
print(dataset)

for i in range(2):
    example = dataset["train"][i]
    print(f"\n--- example {i} ---")
    print("question:", example["question"])
    print("context :", example["context"][:400], "...")
    print("answers :", example["answers"])

Токенизация и построение начальных и конечных позиций.

In [ ]:
checkpoint = "best-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
max_length = 384
doc_stride = 128


def preprocess_qa_train(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = inputs["offset_mapping"]
    sample_map = inputs["overflow_to_sample_mapping"]

    start_positions = []
    end_positinons = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)
        context_start = 0

        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1

        while sequence_ids[context_end] != 1:
            context_end -= 1

        if (
            offsets[context_start][0] > end_char or
            offsets[context_end][1] < start_char
        ):
            start_positions.append(0)
            end_positions.append(0)
        else:
            token_start = context_start

            while (
                token_start <= context_end and
                offsets[token_start][0] < start_char
            ):
                token_start += 1

            start_positions.append(token_start - 1)

            while (
                token_end >= context_start and
                offsets[token_end][1] >= end_char
            ):
                token_end -= 1

            end_positions.append(token_end + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs


train_small = dataset["train"].select(range(3000))
val_small = dataset["validation"].select(range(500))

train_tok = train_small.map(
    preprocess_qa_train,
    batched=True,
    remove_columns=train_small.column_names,
)
val_tok = val_small.map(
    preprocess_qa_train,
    batched=True,
    remove_columns=val_small.column_names,
)

print(train_tok)
print(val_tok)

Загрузка BERT с QA Head и печать модели.

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

print(model)
print("\nQA outputs head:")
print(model.qa_outputs)

Fine-tuning на небольшом подмножестве.

In [ ]:
args = TrainingArguments(
    output_dir="./bert_squad",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

trainer.train()
eval = trainer.evaluate()
eval

Пример инференса.

In [ ]:
example = dataset["validation"][0]
question = example["question"]
context = example["context"]
inputs = tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation=True,
    max_length=384,
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

start_idx = int(outputs.start_logits.argmax(dim=-1)[0])
end_idx = int(outputs.end_logits.argmax(dim=-1)[0])

tokens = inputs["input_ids"][0]
pred_answer = tokenizer.decode(
    tokens[start_idx:end_idx + 1],
    skip_special_tokens=True,
)

print("QUESTION         :", question)
print("PREDICATED ANSWER:", pred_answer)
print("GOLD ANSWERS     :", example["answers"]["text"])

Контрольные вопросы:

1. Почему extractive QA требует start/end logits  
В extractive QA ответ — это непрерывный фрагмент (span) исходного контекста, а не сгенерированный текст и не метка класса. Чтобы указать такой фрагмент, модели достаточно предсказать две вещи: где он начинается и где он заканчивается. Именно это и делают start/end logits — для каждого токена входа модель выдаёт два числа: «насколько вероятно, что ответ начинается на этом токене» и «насколько вероятно, что он здесь заканчивается».

    Дальше применяется softmax по длине последовательности отдельно для start и для end, и ответом считается пара `(i, j)` (где `i ≤ j` и обычно `j − i < max_answer_length`), максимизирующая сумму `start_logit[i] + end_logit[j]`. Два логита нужны именно потому, что span задаётся **двумя границами** — одного числа на токен было бы недостаточно, чтобы однозначно описать отрезок произвольной длины.

    Отдельно стоит упомянуть «нет ответа» (SQuAD 2.0, MuSeRC-подобные задачи): обычно это кодируется как span `(0, 0)`, указывающий на токен `[CLS]` — то есть модель тем же механизмом сообщает, что ответа в контексте нет.

2. Чем QA head отличается от обычной классификации  
У классификационной головы (как в MRPC, QNLI, MuSeRC) логиты считаются один раз на всю последовательность — обычно поверх эмбеддинга `[CLS]` (или pooled output). Размерность выхода — `[batch, num_labels]`, и задача сводится к выбору одного класса из фиксированного множества.

    У QA-головы логиты считаются для каждого токена входа, причём сразу в двух «каналах» — start и end. Размерность выхода — `[batch, seq_len, 2]` (или два тензора `[batch, seq_len]`). Технически это просто линейный слой `hidden_size → 2`, применённый к выходам всех токенов, а не только к `[CLS]`. Ключевые отличия:

    - На что смотрим: классификация — на один агрегированный вектор; QA — на последовательность токенов.
    - Пространство ответов: у классификации оно фиксировано (`num_labels` заранее известно); у QA оно зависит от длины входа — ответом может быть любой из `O(seq_len²)` возможных спанов.
    - Loss: в классификации — одна cross-entropy по меткам классов; в QA — две cross-entropy (по позиции начала и по позиции конца), которые усредняются.
    - Постобработка: у классификации — `argmax` по логитам. У QA нужен отдельный шаг: перебрать top-k пар `(start, end)`, отфильтровать невалидные (end < start, span вне контекста, слишком длинный), спроецировать индексы токенов обратно в символы исходного текста через `offset_mapping`.
    
3. Почему для длинных контекстов нужен `stride`  
Трансформеры-энкодеры имеют жёсткий лимит на длину входа (обычно 512 токенов у BERT-подобных моделей). Контекст в extractive QA часто длиннее — целый абзац из Википедии, документ, параграф MuSeRC. Простое обрезание (`truncation=True`) решает проблему длины, но выбрасывает часть текста, и если правильный ответ оказался в отрезанном куске, модель его физически не увидит и никогда не сможет предсказать.

    Решение — скользящее окно с перекрытием. Длинный контекст разбивается на несколько перекрывающихся «фич», каждая из которых помещается в лимит модели. `stride` задаёт размер перекрытия между соседними окнами (в токенах). В `transformers` это включается так:

    ```python
    tokenizer(
        question, context,
        truncation="only_second",       # режем только контекст, вопрос оставляем целиком
        max_length=384,
        stride=128,                      # соседние окна перекрываются на 128 токенов
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )
    ```

    Зачем именно перекрытие, а не просто нарезка встык: если ответ расположен на границе между двумя окнами, то при нарезке встык он окажется разорван — начало в одном окне, конец в другом — и ни одно из окон не будет содержать полный span. Перекрытие в `stride` токенов гарантирует, что любой ответ короче `stride` целиком поместится хотя бы в одно окно.

    На инференсе модель прогоняется по всем окнам одного примера, логиты собираются вместе, и финальный ответ выбирается как лучший span по всем окнам сразу — с учётом того, какие токены в каждом окне относятся к контексту (а не к вопросу или спецтокенам), это отслеживается через `sequence_ids` и `offset_mapping`.ллоло